# ATDL Final — Tasks 5–12, artifact-first execution

This is the authoritative final notebook. It resumes from verified Tasks 1–4 and executes only production-implemented, validation-only one-seed screens. It never retrains Tasks 1–4, never overwrites an artifact, and never loads the official test set.

**Completed control:** Task 4 vanilla KD, 95.12% ± 0.15% validation (seeds 42/43/44). DKD and DIST are recorded controls below. The remaining executable stages are QFD-style quantized-feature transfer, normalized attention transfer, and RKD-style relational transfer. Each gets one smoke epoch then one 50-epoch seed-42 screen.

## 0. Exact, verified artifact paths

All paths in this notebook are repository-relative and checked before training.

In [1]:
from pathlib import Path
import json, subprocess, sys
PROJECT = Path('/home/vu-lab03-pc17/ATDL-1').resolve()
assert PROJECT.exists()
ARTIFACTS = {
 'teacher': PROJECT/'resnet34_cifar10_fp32_best.pth',
 'fp32_student': PROJECT/'results/best_models/task2_b1_fp32_resnet18_best.pth',
 'ternary_task3': PROJECT/'results/best_models/task3_b2_default_best.pth',
 'task4_summary': PROJECT/'results/task4/task4_b3_final_t2_lam09_aggregate_summary.json',
 'task4_best': PROJECT/'results/task4/best_models/task4_b3_final_t2_lam09_remaining_rerun1_best.pth',
 'dkd_summary': PROJECT/'results/task4/task5_dkd_final_t2_lam09_a1_b8_r1_summary.json',
 'dist_summary': PROJECT/'results/task4/task7_dist_t1_lam05_r1_summary.json',
 'trainer': PROJECT/'scripts/train_student_vanilla_kd.py',
 'pipeline': PROJECT/'scripts/run_final_notebook_pipeline.py',
}
missing = [name for name, path in ARTIFACTS.items() if not path.exists()]
assert not missing, f'Missing required artifact(s): {missing}'
for name, path in ARTIFACTS.items(): print(f'{name:16} {path.relative_to(PROJECT)}')

teacher          resnet34_cifar10_fp32_best.pth
fp32_student     results/best_models/task2_b1_fp32_resnet18_best.pth
ternary_task3    results/best_models/task3_b2_default_best.pth
task4_summary    results/task4/task4_b3_final_t2_lam09_aggregate_summary.json
task4_best       results/task4/best_models/task4_b3_final_t2_lam09_remaining_rerun1_best.pth
dkd_summary      results/task4/task5_dkd_final_t2_lam09_a1_b8_r1_summary.json
dist_summary     results/task4/task7_dist_t1_lam05_r1_summary.json
trainer          scripts/train_student_vanilla_kd.py
pipeline         scripts/run_final_notebook_pipeline.py


## 1. Immutable task history and validation-only firewall

In [2]:
t4 = json.loads(ARTIFACTS['task4_summary'].read_text())
dkd = json.loads(ARTIFACTS['dkd_summary'].read_text())
dist = json.loads(ARTIFACTS['dist_summary'].read_text())
assert t4['training_seeds'] == [42, 43, 44]
assert t4['test_evaluation'] == dkd['test_evaluation'] == dist['test_evaluation'] == 'not_run'
assert t4['all_conv_and_fc_ternary'] and dkd['all_conv_and_fc_ternary'] and dist['all_conv_and_fc_ternary']
print({'Task4_vanilla_mean': t4['validation_mean'], 'Task4_vanilla_std': t4['validation_std'],
       'DKD_3seed_mean': dkd['validation_mean'], 'DIST_one_seed': dist['validation_mean'],
       'official_test': 'LOCKED_NOT_RUN'})

{'Task4_vanilla_mean': 0.9512, 'Task4_vanilla_std': 0.0015099668870541527, 'DKD_3seed_mean': 0.9488, 'DIST_one_seed': 0.9502, 'official_test': 'LOCKED_NOT_RUN'}


## 2. Production implementations and method mapping

The notebook delegates all training to the shared project trainer. QFD matches a frozen ternarized teacher terminal feature; AT matches normalized terminal attention maps; RKD matches scale-normalized penultimate pairwise distances. Every method retains the Task-4 teacher, warm start, frozen split, QAT, optimizer, and strict all-layer ternary deployment constraint.

In [3]:
STAGES = [
 {'task':'T6', 'method':'qfd', 'config':'configs/kd/resnet18_ternary_qfd_screen.yaml', 'run':'task8_qfd_screen_t2_lam09_aux1_r1'},
 {'task':'T7', 'method':'at',  'config':'configs/kd/resnet18_ternary_at_screen.yaml',  'run':'task9_at_screen_t2_lam09_aux1_r1'},
 {'task':'T8', 'method':'rkd', 'config':'configs/kd/resnet18_ternary_rkd_screen.yaml', 'run':'task10_rkd_screen_t2_lam09_aux1_r1'},
]
for stage in STAGES:
    p = PROJECT/stage['config']; assert p.exists(), p
    print(stage)
# Tasks 9–12 are diagnostics/decision/reporting stages; they consume saved histories and never call a test loader.

{'task': 'T6', 'method': 'qfd', 'config': 'configs/kd/resnet18_ternary_qfd_screen.yaml', 'run': 'task8_qfd_screen_t2_lam09_aux1_r1'}
{'task': 'T7', 'method': 'at', 'config': 'configs/kd/resnet18_ternary_at_screen.yaml', 'run': 'task9_at_screen_t2_lam09_aux1_r1'}
{'task': 'T8', 'method': 'rkd', 'config': 'configs/kd/resnet18_ternary_rkd_screen.yaml', 'run': 'task10_rkd_screen_t2_lam09_aux1_r1'}


## 3. Consistency gate

Fail closed if a production training source imports a test loader or if the exact checkpoint/model identities are wrong.

In [4]:
import torch
forbidden = ('get_test_loader(', 'CIFAR10Test', 'test_loader =')
for source in [ARTIFACTS['trainer'], ARTIFACTS['pipeline']]:
    hits = [token for token in forbidden if token in source.read_text()]
    assert not hits, f'Test-firewall violation: {source}: {hits}'
teacher = torch.load(ARTIFACTS['teacher'], map_location='cpu', weights_only=False)
student = torch.load(ARTIFACTS['fp32_student'], map_location='cpu', weights_only=False)
assert teacher['arch'] == 'ResNet34-CIFAR' and teacher['num_classes'] == 10
assert student['arch'] == 'ResNet18-CIFAR' and student['num_classes'] == 10
print('PASS: paths, architectures, test firewall, and prior artifacts are consistent.')

PASS: paths, architectures, test firewall, and prior artifacts are consistent.


## 4. Training — sequential, one seed per method

Set `RUN_PIPELINE=True` to execute QFD → AT → RKD. Each stage runs a one-epoch smoke gate before its 50-epoch seed-42 screen. Existing summaries are skipped, so this cell is safe to resume. The GPU is never shared with another run.

In [5]:
RUN_PIPELINE = True
command = [sys.executable, str(ARTIFACTS['pipeline'])]
print('Pipeline command:', ' '.join(command))
if RUN_PIPELINE:
    completed = subprocess.run(command, cwd=PROJECT)
    assert completed.returncode == 0, 'Final notebook pipeline failed; inspect preserved artifacts before retrying.'
else:
    print('Training disabled by configuration.')

Pipeline command: /home/vu-lab03-pc17/ATDL-1/.venv/bin/python /home/vu-lab03-pc17/ATDL-1/scripts/run_final_notebook_pipeline.py


Task 4 validation-only: teacher=resnet34_cifar10_fp32_best.pth SHA256=ed19ff5fa087…

=== task8_qfd_smoke_r1 seed 42 | T=2, lambda=0.9 | 1 epochs ===


T4 S42 1/1:   0%|          | 0/352 [00:00<?, ?it/s]

T4 S42 1/1:   1%|          | 4/352 [00:01<01:56,  2.99it/s]

T4 S42 1/1:   3%|▎         | 10/352 [00:01<00:41,  8.20it/s]

...

T4 S42 50/50:  95%|█████████▍| 334/352 [00:11<00:00, 29.65it/s]

T4 S42 50/50:  97%|█████████▋| 340/352 [00:12<00:00, 29.68it/s]

T4 S42 50/50:  98%|█████████▊| 346/352 [00:12<00:00, 29.62it/s]

S42 E 50/50 total=0.0530 CE=0.5011 KD=0.0032 val=94.92% lr=0.000122
Task 4 validation: 95.12% ± 0.00%
Summary: /home/vu-lab03-pc17/ATDL-1/results/task4/task10_rkd_screen_t2_lam09_aux1_r1_summary.json


{
  "status": "COMPLETE",
  "screens": {
    "qfd": {
      "validation_mean": 0.95,
      "validation_best": 0.95,
      "summary": "results/task4/task8_qfd_screen_t2_lam09_aux1_r1_summary.json",
      "status": "one_seed_provisional"
    },
    "at": {
      "validation_mean": 0.9502,
      "validation_best": 0.9502,
      "summary": "results/task4/task9_at_screen_t2_lam09_aux1_r1_summary.json",
      "status": "one_seed_provisional"
    },
    "rkd": {
      "validation_mean": 0.9512,
      "validation_best": 0.9512,
      "summary": "results/task4/task10_rkd_screen_t2_lam09_aux1_r1_summary.json",
      "status": "one_seed_provisional"
    }
  },
  "best_screen": "rkd",
  "official_test": "not_run"
}


## 5. Tasks 9–12 — evidence, diagnostics, compression, and final decision

One-seed screens are provisional evidence only. The final method remains the matched three-seed Task-4 vanilla-KD control unless a later independently authorized confirmation exceeds it. The official test set remains locked.

In [6]:
summaries = {'Task4': ARTIFACTS['task4_summary'], 'DKD': ARTIFACTS['dkd_summary'], 'DIST': ARTIFACTS['dist_summary']}
for stage in STAGES:
    p = PROJECT/'results/task4'/f"{stage['run']}_summary.json"
    if p.exists(): summaries[stage['method'].upper()] = p
rows = {}
for name, path in summaries.items():
    data = json.loads(path.read_text()); rows[name] = {'validation_mean': data['validation_mean'], 'validation_std': data['validation_std'], 'best': data['validation_best'], 'test': data['test_evaluation']}
print(json.dumps(rows, indent=2))
assert all(row['test'] == 'not_run' for row in rows.values())
print('Final decision rule: Task 4 three-seed mean is the control; screens are not promoted without confirmation.')

{
  "Task4": {
    "validation_mean": 0.9512,
    "validation_std": 0.0015099668870541527,
    "best": 0.9526,
    "test": "not_run"
  },
  "DKD": {
    "validation_mean": 0.9488,
    "validation_std": 0.0015999999999999903,
    "best": 0.9504,
    "test": "not_run"
  },
  "DIST": {
    "validation_mean": 0.9502,
    "validation_std": 0.0,
    "best": 0.9502,
    "test": "not_run"
  },
  "QFD": {
    "validation_mean": 0.95,
    "validation_std": 0.0,
    "best": 0.95,
    "test": "not_run"
  },
  "AT": {
    "validation_mean": 0.9502,
    "validation_std": 0.0,
    "best": 0.9502,
    "test": "not_run"
  },
  "RKD": {
    "validation_mean": 0.9512,
    "validation_std": 0.0,
    "best": 0.9512,
    "test": "not_run"
  }
}
Final decision rule: Task 4 three-seed mean is the control; screens are not promoted without confirmation.


## 6. End state

This notebook saves all training artifacts through the production trainer and writes `research_state_checkpoint.json` through its controller. It is resumable and leaves deep AutoResearch preserved rather than abandoned.